In [2]:
!git clone https://github.com/tarun1125/CSAIML-Capstone-Project-20.git

Cloning into 'CSAIML-Capstone-Project-20'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 56 (delta 10), reused 42 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 26.10 KiB | 8.70 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [3]:
%pip install transformers accelerate torch bitsandbytes sentencepiece pymongo huggingface_hub codebleu bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.2/546.2 kB 41.8 MB/s eta 0:00:00


In [8]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [9]:
cd /content/CSAIML-Capstone-Project-20

/content/CSAIML-Capstone-Project-20


In [10]:
import os

from google.colab import userdata
HF_TOKEN   = userdata.get("HF_TOKEN")
ATLAS_URI  = userdata.get("MONGODB_URI")

import logging
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("capstone-eval")


In [ ]:
SYSTEM_PROMPT = """You are a MongoDB query expert.
When given a natural-language question and a database schema, you output ONLY the raw PyMongo query — no explanation, no markdown, no prose.

Schema:
- singers:    { singer_id, name, country, age }
- concerts:   { concert_id, concert_name, theme, stadium_id, year (string) }
- stadiums:   { stadium_id, name, capacity, city }
- singer_concerts: { singer_id, concert_id }
- pets:       { pet_id, pet_type, pet_age, weight }
- students:   { stu_id, fname, lname, age, sex }
- has_pet:    { stu_id, pet_id }
- cars:       { car_id, maker, model, model_year, horsepower, mpg, origin }
- highschoolers: { id, name, grade }
- likes:      { student_id, liked_id }
- friends:    { student_id, friend_id }

Rules:
1. Output ONLY the PyMongo expression (e.g. list(db.singers.find({...})))
2. Use db.<collection>.<method>() syntax
3. Do NOT wrap in ```python or any markdown
4. Do NOT add any explanation before or after"""

QUERIES = [
    ("easy-1",    "How many singers are there?"),
    ("easy-2",    "What are the names of singers from France?"),
    ("easy-3",    "Find the maximum weight of all pets."),
    ("medium-1",  "What are the names and ages of singers, ordered by age?"),
    ("medium-2",  "How many concerts were held in each stadium?"),
    ("medium-3",  "Find pet types that have more than one pet, with their counts."),
    ("high-1",    "What are the names of singers who performed in a concert in 2014?"),
    ("high-2",    "Find the first names of students who have a dog as a pet."),
    ("high-3",    "For each country of origin, find the average miles per gallon of cars."),
    ("complex-1", "Find singers who have not participated in any concert."),
    ("complex-2", "Find students who have more friends than the average number of friends per student."),
    ("complex-3", "For each maker, find the model with the highest horsepower."),
]

predictions_qwen = []

for qid, nl in QUERIES:
    log.info("[%s] querying Qwen...", qid)
    t0 = time.time()

    try:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": nl}
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.0,
            do_sample=False
        )

        generated = outputs[0][inputs.input_ids.shape[1]:]
        raw = tokenizer.decode(generated, skip_special_tokens=True).strip()

        raw = raw.replace("```python", "").replace("```", "").strip()

        latency = time.time() - t0
        log.info("[%s] OK (%.2fs): %s", qid, latency, raw[:80])

        predictions_qwen.append({
            "id": qid,
            "question": nl,
            "generated_query": raw,
            "latency_s": round(latency, 3)
        })

    except Exception as e:
        log.error("[%s] FAILED: %s", qid, e)
        predictions_qwen.append({
            "id": qid,
            "generated_query": "",
            "error": str(e)
        })

with open("qwen2.5_coder_results.json", "w") as f:
    json.dump(predictions_qwen, f, indent=2)

log.info("Saved qwen2.5_coder_results.json")

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
